In [103]:
import pandas as pd
import numpy as np


ruta_datos = r"C:\Users\LauraB\Desktop\BDATA 2\RETO 05\RETO5\data\datos_preprocesados.csv"

ruta_clasificacion = r"C:\Users\LauraB\Desktop\BDATA 2\RETO 05\RETO5\data\clientes_clasificados_frances_aleman.csv"

ruta_predicciones = r"C:\Users\LauraB\Desktop\BDATA 2\RETO 05\RETO5\data\predicciones_impago.csv"


datos = pd.read_csv(ruta_datos)
clasificacion = pd.read_csv(ruta_clasificacion)
predicciones = pd.read_csv(ruta_predicciones)


columnas_clasificacion = [
    "ID",
    "Esfuerzo_Inicial_Fr",
    "Esfuerzo_Inicial_Al",
    "Ahorro_Intereses_Al",
    "Intereses_Totales_Fr",
    "Sobreesfuerzo_Inicial_Al",
    "Sistema_Recomendado",
    "Ambos_Superan_Limite",
    "Diferencia_Esfuerzo",
    "Ahorro_Intereses_Pct"
]

clasificacion = clasificacion[columnas_clasificacion]


df = datos.merge(clasificacion, on="ID", how="inner")
df = df.merge(predicciones, on="ID", how="inner")


print("Número de préstamos:", len(df))
print("Número de identificadores únicos:", df["ID"].nunique())

display(df.head())

Número de préstamos: 178606
Número de identificadores únicos: 178606


,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Esfuerzo_Inicial_Fr,Esfuerzo_Inicial_Al,Ahorro_Intereses_Al,Intereses_Totales_Fr,Sobreesfuerzo_Inicial_Al,Sistema_Recomendado,Ambos_Superan_Limite,Diferencia_Esfuerzo,Ahorro_Intereses_Pct,PD_estimada
0,KG9CT9,34,56572,73978,763,86,1,13.91,60,0.55,...,36.439698,44.343385,2940.716995,29095.330653,372.606139,Francés,True,7.903687,10.107178,0.616707
1,V5O0K9,37,68698,150000,824,162,2,10.53,24,0.37,...,121.549207,132.165420,550.623822,17003.748822,607.760466,Francés,True,10.616212,3.238250,0.397428
2,YE5GD0,40,50000,7490,671,167,2,9.24,60,0.10,...,3.752495,4.380152,132.211310,1891.237810,26.152370,Alemán,False,0.627657,6.990729,0.455489
3,Q6HQ3D,38,63033,11801,649,196,1,7.63,48,0.10,...,5.445751,6.108969,91.132177,1929.482123,34.837147,Alemán,False,0.663217,4.723142,0.412927
4,XO8CO4,19,22301,40000,788,17,4,8.40,36,0.50,...,67.845326,74.854640,210.558419,5390.558419,130.262266,Francés,True,7.009314,3.906059,0.565416


In [104]:
prevalencia_real = df["Impago"].mean()

pd_original = df["PD_estimada"].clip(0.000001, 0.999999)

odds = pd_original / (1 - pd_original)

odds_ajustadas = odds * (
    prevalencia_real / (1 - prevalencia_real)
)

df["PD_ajustada"] = odds_ajustadas / (1 + odds_ajustadas)

print("Impago real:", round(prevalencia_real * 100, 2), "%")
print("PD media original:", round(df["PD_estimada"].mean() * 100, 2), "%")
print("PD media ajustada:", round(df["PD_ajustada"].mean() * 100, 2), "%")

Impago real: 11.63 %
PD media original: 45.86 %
PD media ajustada: 11.63 %


In [105]:
TASA_DESCUENTO_ANUAL = 0.04
LGD = 0.60

tasa_descuento_mensual = (
    (1 + TASA_DESCUENTO_ANUAL) ** (1 / 12) - 1
)

capital = df["Monto_Inicial"].to_numpy(dtype=float)

duracion = df["Duracion"].to_numpy(dtype=int)

interes_mensual = (
    df["Ratio_Interes"].to_numpy(dtype=float) / 100 / 12
)

pd_estimada = df["PD_ajustada"].to_numpy(dtype=float)

hazard_mensual = 1 - (1 - pd_estimada) ** (1 / duracion)

print("Capital total:", round(capital.sum(), 2), "€")
print("Duración máxima:", duracion.max(), "meses")
print("LGD:", LGD * 100, "%")
print("Tasa de descuento anual:", TASA_DESCUENTO_ANUAL * 100, "%")


Capital total: 8071341941.0 €
Duración máxima: 60 meses
LGD: 60.0 %
Tasa de descuento anual: 4.0 %


In [106]:
cuota_frances = np.where(
    interes_mensual == 0,
    capital / duracion,
    capital * interes_mensual /
    (1 - (1 + interes_mensual) ** (-duracion))
)

In [107]:
amortizacion_alemana = capital / duracion

print("Cuota francesa total:", round(cuota_frances.sum(), 2), "€")
print(
    "Primera cuota alemana total:",
    round((amortizacion_alemana + capital * interes_mensual).sum(), 2),
    "€"
)

Cuota francesa total: 306522366.9 €
Primera cuota alemana total: 339046029.23 €


In [108]:
saldo_frances = capital.copy()
saldo_aleman = capital.copy()

supervivencia = np.ones(len(capital))

meses_maximos = int(duracion.max())

flujos_frances = np.zeros(meses_maximos + 1)
flujos_aleman = np.zeros(meses_maximos + 1)

flujos_frances[0] = -capital.sum()
flujos_aleman[0] = -capital.sum()


In [109]:
for mes in range(1, meses_maximos + 1):

    indices = np.where(duracion >= mes)[0]

    if len(indices) == 0:
        continue

    prob_impago = supervivencia[indices] * hazard_mensual[indices]
    prob_cobro = supervivencia[indices] * (1 - hazard_mensual[indices])

    # SISTEMA FRANCES

    saldo_fr = saldo_frances[indices]
    interes_fr = saldo_fr * interes_mensual[indices]

    amortizacion_fr = np.minimum(
        saldo_fr,
        cuota_frances[indices] - interes_fr
    )

    ultimo_mes = duracion[indices] == mes
    amortizacion_fr[ultimo_mes] = saldo_fr[ultimo_mes]

    cuota_fr = amortizacion_fr + interes_fr

    flujo_fr = (
        prob_cobro * cuota_fr
        + prob_impago * (1 - LGD) * saldo_fr
    )

    flujos_frances[mes] = flujo_fr.sum()

    saldo_frances[indices] = np.maximum(
        0, saldo_fr - amortizacion_fr
    )

    # SISTEMA ALEMAN

    saldo_al = saldo_aleman[indices]

    amortizacion_al = np.minimum(
        saldo_al,
        amortizacion_alemana[indices]
    )

    amortizacion_al[ultimo_mes] = saldo_al[ultimo_mes]

    interes_al = saldo_al * interes_mensual[indices]

    cuota_al = amortizacion_al + interes_al

    flujo_al = (
        prob_cobro * cuota_al
        + prob_impago * (1 - LGD) * saldo_al
    )

    flujos_aleman[mes] = flujo_al.sum()

    saldo_aleman[indices] = np.maximum(
        0, saldo_al - amortizacion_al
    )

    

    supervivencia[indices] *= (
        1 - hazard_mensual[indices]
    )

print("Flujos mensuales calculados correctamente.")


Flujos mensuales calculados correctamente.


In [111]:
descuentos = (
    1 + tasa_descuento_mensual
) ** np.arange(meses_maximos + 1)


VAN_frances = np.sum(flujos_frances / descuentos)

VAN_aleman = np.sum(flujos_aleman / descuentos)

print("VAN del sistema francés:", round(VAN_frances, 2), "€")
print("VAN del sistema alemán:", round(VAN_aleman, 2), "€")

VAN del sistema francés: 581835811.02 €
VAN del sistema alemán: 541599625.92 €


In [112]:
def calcular_tir(flujos):

    inferior = -0.99
    superior = 0.01

    
    def calcular_van(tasa):
        van = 0

        for t in range(len(flujos)):
            van += flujos[t] / (1 + tasa) ** t

        return van

    
    while calcular_van(superior) > 0:
        superior = superior * 2 + 0.01

        if superior > 100:
            return np.nan

    
    for _ in range(1000):

        medio = (inferior + superior) / 2
        van_medio = calcular_van(medio)

        if abs(van_medio) < 0.01:
            return medio

        if van_medio > 0:
            inferior = medio
        else:
            superior = medio

    return (inferior + superior) / 2


In [113]:
tir_mensual_frances = calcular_tir(flujos_frances)
tir_mensual_aleman = calcular_tir(flujos_aleman)

# Convertir a TIR anual efectiva
tir_anual_frances = (1 + tir_mensual_frances) ** 12 - 1
tir_anual_aleman = (1 + tir_mensual_aleman) ** 12 - 1

# Mostrar resultados
print("TIR mensual francés:", round(tir_mensual_frances * 100, 4), "%")
print("TIR anual francés:", round(tir_anual_frances * 100, 4), "%")

print("\nTIR mensual alemán:", round(tir_mensual_aleman * 100, 4), "%")
print("TIR anual alemán:", round(tir_anual_aleman * 100, 4), "%")

TIR mensual francés: 0.7132 %
TIR anual francés: 8.9028 %

TIR mensual alemán: 0.7074 %
TIR anual alemán: 8.827 %


In [114]:
resultados = pd.DataFrame({
    "Indicador": [
        "VAN (€)",
        "TIR mensual (%)",
        "TIR anual (%)"
    ],
    "Sistema francés": [
        VAN_frances,
        tir_mensual_frances * 100,
        tir_anual_frances * 100
    ],
    "Sistema alemán": [
        VAN_aleman,
        tir_mensual_aleman * 100,
        tir_anual_aleman * 100
    ]
})

display(resultados.round(4))

diferencia_van = VAN_aleman - VAN_frances

print(
    "Diferencia de VAN entre ambos sistemas:",
    round(diferencia_van, 2),
    "€"
)

,Indicador,Sistema francés,Sistema alemán
0,VAN (€),5.818358e+08,5.415996e+08
1,TIR mensual (%),7.132000e-01,7.074000e-01
2,TIR anual (%),8.902800e+00,8.827000e+00


Diferencia de VAN entre ambos sistemas: -40236185.1 €


In [116]:
resultados.to_csv(
    "calculo_VAN_TIR.csv",
    index=False,
    encoding="utf-8-sig"
)


tabla_flujos = pd.DataFrame({
    "Mes": np.arange(meses_maximos + 1),
    "Flujo_Frances": flujos_frances,
    "Flujo_Aleman": flujos_aleman
})

tabla_flujos.to_csv(
    "flujos_mensuales.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos exportados correctamente.")

Archivos exportados correctamente.
